## Load silver dataset

In [0]:
from pyspark.sql.functions import col, when, avg, round as spark_round

silver = spark.table("silver_valuation_multiple")
print(f"Total silver rows: {silver.count()}")

Total silver rows: 404


## Gold Table 1 — Industry Valuation Benchmark
### table name : gold_industry_valuation 

In [0]:
bracket_data = silver.filter(col("ev_bracket").isNotNull())

ebitda = bracket_data.filter(col("metric_type") == "ev_ebitda") \
    .select("sub_vertical", "ev_bracket",
            col("median_multiple").alias("median_ev_ebitda"),
            col("p25").alias("p25_ev_ebitda"),
            col("p75").alias("p75_ev_ebitda"),
            col("deal_count").alias("deal_count_ebitda"))

revenue = bracket_data.filter(col("metric_type") == "ev_revenue") \
    .select("sub_vertical", "ev_bracket",
            col("median_multiple").alias("median_ev_revenue"),
            col("deal_count").alias("deal_count_revenue"))

gold_industry_valuation = ebitda.join(revenue, ["sub_vertical", "ev_bracket"], "outer") \
    .withColumn("deal_count",
                when(col("deal_count_ebitda").isNotNull(), col("deal_count_ebitda"))
                .otherwise(col("deal_count_revenue"))) \
    .select("sub_vertical", "ev_bracket", "median_ev_ebitda", "median_ev_revenue",
            "p25_ev_ebitda", "p75_ev_ebitda", "deal_count")

display(gold_industry_valuation.limit(10))
print(f"Rows: {gold_industry_valuation.count()}")

sub_vertical,ev_bracket,median_ev_ebitda,median_ev_revenue,p25_ev_ebitda,p75_ev_ebitda,deal_count
advertising-agency,25m_100m_ev,null,1.75,null,null,11
advertising-agency,5m_25m_ev,null,1.29,null,null,14
aerospace,100m_500m_ev,null,2.38,null,null,24
aerospace,25m_100m_ev,null,1.86,null,null,13
aerospace,5m_25m_ev,null,0.77,null,null,10
aerospace,over_500m_ev,14.25,2.12,12.9,16.92,24
ambulatory-surgery-center,25m_100m_ev,9.5,2.5,6.8,12.2,11
ambulatory-surgery-center,5m_25m_ev,8.1,1.4,5.47,11.38,20
ambulatory-surgery-center,under_5m_ev,null,0.9,null,null,15
apparel,100m_500m_ev,null,0.8,null,null,15


Rows: 140


In [0]:
gold_industry_valuation_path = "abfss://gold@stmavaluationplatform.dfs.core.windows.net/gold_industry_valuation"

gold_industry_valuation.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(gold_industry_valuation_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_industry_valuation
USING DELTA
LOCATION '{gold_industry_valuation_path}'
""")

print(f"Written {gold_industry_valuation.count()} rows")

Written 140 rows


## Gold Table 2 — Valuation Trend 
### table name : gold_valuation_trend 

In [0]:
year_data = silver.filter(col("source_year").isNotNull())

ebitda_y = year_data.filter(col("metric_type") == "ev_ebitda") \
    .select("sub_vertical", col("source_year").alias("year"),
            col("median_multiple").alias("median_ev_ebitda"),
            col("deal_count").alias("deal_count_ebitda"))

revenue_y = year_data.filter(col("metric_type") == "ev_revenue") \
    .select("sub_vertical", col("source_year").alias("year"),
            col("median_multiple").alias("median_ev_revenue"),
            col("deal_count").alias("deal_count_revenue"))

gold_valuation_trend = ebitda_y.join(revenue_y, ["sub_vertical", "year"], "outer") \
    .withColumn("deal_count",
                when(col("deal_count_ebitda").isNotNull(), col("deal_count_ebitda"))
                .otherwise(col("deal_count_revenue"))) \
    .withColumn("primary_multiple",
                when(col("median_ev_ebitda").isNotNull(), col("median_ev_ebitda"))
                .otherwise(col("median_ev_revenue")))

display(gold_valuation_trend.limit(10))
print(f"Rows: {gold_valuation_trend.count()}")

sub_vertical,year,median_ev_ebitda,deal_count_ebitda,median_ev_revenue,deal_count_revenue,deal_count,primary_multiple
aerospace,2018,null,null,1.89,11,11,1.89
aerospace,2019,null,null,1.64,13,13,1.64
aerospace,2021,null,null,2.34,13,13,2.34
aerospace,2024,null,null,2.27,17,17,2.27
ambulatory-surgery-center,2021,null,null,1.9,10,10,1.9
ambulatory-surgery-center,2022,null,null,1.25,10,10,1.25
ambulatory-surgery-center,2024,null,null,2.1,12,12,2.1
apparel,2018,null,null,0.62,13,13,0.62
apparel,2019,null,null,0.67,13,13,0.67
auto-parts,2022,null,null,0.85,12,12,0.85


Rows: 139


### calculate yoy change (year over year , per sub vertical, exactly 1 year gap) 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

trend_window = Window.partitionBy("sub_vertical").orderBy("year")

gold_valuation_trend = gold_valuation_trend \
    .withColumn("prev_year", lag("year").over(trend_window)) \
    .withColumn("prev_year_multiple", lag("primary_multiple").over(trend_window)) \
    .withColumn(
        "yoy_change",
        when(
            (col("prev_year_multiple").isNotNull()) &
            (col("prev_year_multiple") != 0) &
            (col("year") - col("prev_year") == 1),
            spark_round(((col("primary_multiple") - col("prev_year_multiple")) / col("prev_year_multiple")) * 100, 2)
        ).otherwise(None)
    ).select("year", "sub_vertical", "median_ev_ebitda", "median_ev_revenue", "deal_count", "yoy_change")

display(gold_valuation_trend.orderBy("sub_vertical", "year").limit(15))
print(f"Rows: {gold_valuation_trend.count()}")

year,sub_vertical,median_ev_ebitda,median_ev_revenue,deal_count,yoy_change
2018,aerospace,null,1.89,11,null
2019,aerospace,null,1.64,13,-13.23
2021,aerospace,null,2.34,13,null
2024,aerospace,null,2.27,17,null
2021,ambulatory-surgery-center,null,1.9,10,null
2022,ambulatory-surgery-center,null,1.25,10,-34.21
2024,ambulatory-surgery-center,null,2.1,12,null
2018,apparel,null,0.62,13,null
2019,apparel,null,0.67,13,8.06
2022,auto-parts,null,0.85,12,null


Rows: 139


In [0]:
gold_valuation_trend_path = "abfss://gold@stmavaluationplatform.dfs.core.windows.net/gold_valuation_trend"

gold_valuation_trend.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(gold_valuation_trend_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_valuation_trend
USING DELTA
LOCATION '{gold_valuation_trend_path}'
""")

print(f"Written {gold_valuation_trend.count()} rows")

Written 139 rows


## Gold Table 3 — Industry Ranking
### table name : gold_industry_ranking 

In [0]:
from pyspark.sql.functions import rank, percent_rank, round as spark_round

ranking_base = ebitda_y.filter(col("median_ev_ebitda").isNotNull())

rank_window = Window.partitionBy("year").orderBy(col("median_ev_ebitda").desc())

gold_industry_ranking = ranking_base \
    .withColumn("rank", rank().over(rank_window)) \
    .withColumn("percentile", spark_round(100 - (percent_rank().over(rank_window) * 100), 2)) \
    .select(
        "year",
        col("sub_vertical").alias("industry"),
        "median_ev_ebitda",
        "rank",
        "percentile",
        col("deal_count_ebitda").alias("deal_count")
    )

display(gold_industry_ranking.orderBy("year", "rank").limit(15))
print(f"Rows: {gold_industry_ranking.count()}") 

year,industry,median_ev_ebitda,rank,percentile,deal_count
2018,saas,18.15,1,100.0,14
2018,restaurant-qsr,15.99,2,90.0,11
2018,medical-devices,15.63,3,80.0,13
2018,food-manufacturing,15.3,4,70.0,10
2018,oil-gas-services,10.66,5,60.0,50
2018,wholesale-distribution,10.07,6,50.0,14
2018,mental-health,9.95,7,40.0,10
2018,industrial-equipment,9.93,8,30.0,11
2018,it-services,9.41,9,20.0,13
2018,gaming,9.0,10,10.0,11


Rows: 59


In [0]:
gold_industry_ranking_path = "abfss://gold@stmavaluationplatform.dfs.core.windows.net/gold_industry_ranking"

gold_industry_ranking.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(gold_industry_ranking_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_industry_ranking
USING DELTA
LOCATION '{gold_industry_ranking_path}'
""")

print(f"Written {gold_industry_ranking.count()} rows")

Written 59 rows


## Gold Table 4 — EV Bracket Analysis
### table name : gold_ev_bracket_analysis

In [0]:
industry_avg = gold_industry_valuation.groupBy("sub_vertical") \
    .agg(avg("median_ev_ebitda").alias("avg_median_ev_ebitda"))

gold_ev_bracket_analysis = gold_industry_valuation.join(industry_avg, "sub_vertical") \
    .withColumn(
        "valuation_premium",
        when(
            col("avg_median_ev_ebitda").isNotNull() & (col("avg_median_ev_ebitda") != 0),
            spark_round(((col("median_ev_ebitda") - col("avg_median_ev_ebitda")) / col("avg_median_ev_ebitda")) * 100, 2)
        ).otherwise(None)
    ).select("ev_bracket", "sub_vertical", "median_ev_ebitda", "median_ev_revenue",
             "deal_count", "valuation_premium")

display(gold_ev_bracket_analysis.orderBy("sub_vertical", "ev_bracket").limit(15))
print(f"Rows: {gold_ev_bracket_analysis.count()}")

ev_bracket,sub_vertical,median_ev_ebitda,median_ev_revenue,deal_count,valuation_premium
25m_100m_ev,advertising-agency,null,1.75,11,null
5m_25m_ev,advertising-agency,null,1.29,14,null
100m_500m_ev,aerospace,null,2.38,24,null
25m_100m_ev,aerospace,null,1.86,13,null
5m_25m_ev,aerospace,null,0.77,10,null
over_500m_ev,aerospace,14.25,2.12,24,0.0
25m_100m_ev,ambulatory-surgery-center,9.5,2.5,11,7.95
5m_25m_ev,ambulatory-surgery-center,8.1,1.4,20,-7.95
under_5m_ev,ambulatory-surgery-center,null,0.9,15,null
100m_500m_ev,apparel,null,0.8,15,null


Rows: 140


In [0]:
gold_ev_bracket_analysis_path = "abfss://gold@stmavaluationplatform.dfs.core.windows.net/gold_ev_bracket_analysis"

gold_ev_bracket_analysis.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(gold_ev_bracket_analysis_path)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_ev_bracket_analysis
USING DELTA
LOCATION '{gold_ev_bracket_analysis_path}'
""")

print(f"Written {gold_ev_bracket_analysis.count()} rows")

Written 140 rows


## confirming all gold tables


In [0]:
display(spark.sql("SHOW TABLES IN default LIKE 'gold_*'"))

database,tableName,isTemporary
default,gold_data_quality,false
default,gold_ev_bracket_analysis,false
default,gold_industry_ranking,false
default,gold_industry_valuation,false
default,gold_valuation_trend,false
